# SVM train with embeddings

## Caricamento del dataset

In [1]:
import torch
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

In [ ]:
# Carica il dataset
features_path = "/book_jeunesse_features.pt"
loaded = torch.load(features_path)

# Trasformiamo in DataFrame più comodo
df = pd.DataFrame(loaded)

In [3]:
df.head()

,file_name,char_name,char_id,type,Gender,agent_lemmas,patient_lemmas,mod_lemmas,pos_lemmas,agent_embeddings,patient_embeddings,mod_embeddings,pos_embeddings
0,1833_Girardin-Delphine-de_Contes-d-une-vieille...,léon,0,1.0,0.0,"[venir, obtenir, aimer, sauter, mettre, aperce...","[récompenser, emmener, laisse, décider, suivre...","[ravir, volant, volant, volant, affreux, gros,...","[mère, ordinaire, incertitude, mère, course, i...","[[tensor(-0.2522), tensor(-0.2233), tensor(-0....","[[tensor(0.1801), tensor(-0.3205), tensor(-0.2...","[[tensor(-0.2756), tensor(0.4234), tensor(-0.0...","[[tensor(0.0378), tensor(-0.3824), tensor(-0.0..."
1,1833_Girardin-Delphine-de_Contes-d-une-vieille...,zoé,1,1.0,1.0,"[repartir, lever, descendre, apercevoir, doute...","[séparer, aider, regarder, accabler, admirer, ...","[petit, effronté, coquet, petit, malheureux, p...","[face, caprice, robe, amie, mère, crainte, coq...","[[tensor(0.0616), tensor(-0.2516), tensor(-0.0...","[[tensor(0.1705), tensor(-0.2149), tensor(-0.3...","[[tensor(0.0626), tensor(-0.2282), tensor(-0.2...","[[tensor(-0.1416), tensor(0.1424), tensor(-0.3..."
2,1833_Girardin-Delphine-de_Contes-d-une-vieille...,césaro,2,1.0,0.0,"[avoir, dire, aller, dire, lever, faire, écrie...","[rendre, écouter, corriger, fatiguer, prendre,...","[petit, fille, spirituel, petit, âgé, fier, fi...","[défaut, discours, père, langage, femme, père,...","[[tensor(-0.0889), tensor(-0.2208), tensor(-0....","[[tensor(0.5683), tensor(-0.0505), tensor(-0.4...","[[tensor(-0.1546), tensor(-0.4502), tensor(-0....","[[tensor(0.6686), tensor(-0.0990), tensor(0.00..."
3,1833_Girardin-Delphine-de_Contes-d-une-vieille...,aglaure,3,1.0,1.0,"[faire, jouer, devenir, sentir, vouloir, espér...","[appeler, inviter, intimider, voir, environner...","[bonhomme, tout, ancien, ravir, étonné, ingrat...","[spéculation, camarade, habitude, ami, étable,...","[[tensor(-0.1492), tensor(-0.3598), tensor(-0....","[[tensor(-0.1114), tensor(0.1678), tensor(-0.2...","[[tensor(0.4053), tensor(0.0214), tensor(-0.14...","[[tensor(0.0398), tensor(0.1744), tensor(-0.73..."
4,1833_Girardin-Delphine-de_Contes-d-une-vieille...,grignotte,4,2.0,0.0,"[avoir, donner, parler, devoir, moque, avoir, ...","[diriger, voler, retrouver, racheter, disséque...","[grand, grand, pauvre, merveille, léger, joli,...","[an, oncle, fusil, oncle, fusil, cravate, chie...","[[tensor(-0.3065), tensor(-0.5236), tensor(-0....","[[tensor(0.1391), tensor(0.2415), tensor(-0.60...","[[tensor(0.1163), tensor(-1.1308), tensor(-0.0...","[[tensor(-0.1025), tensor(-0.3933), tensor(-0...."


In [4]:
# Filtra solo type 0 e 1 (non mi interessano gli altri tipi di personaggi)
df = df[df['type'].isin([0, 1])].reset_index(drop=True)
print(f"Numero di personaggi filtrati: {len(df)}")

Numero di personaggi filtrati: 546


## Media concatenata di ciascun tipo di embedding

In [5]:
def mean_concat_embeddings(row):
    # Converte ogni tensor torch in numpy
    agent = row['agent_embeddings'].mean(dim=0).numpy() if len(row['agent_embeddings']) > 0 else np.zeros(row['agent_embeddings'].shape[1])
    #patient = row['patient_embeddings'].mean(dim=0).numpy() if len(row['patient_embeddings']) > 0 else np.zeros(row['patient_embeddings'].shape[1])
    mod = row['mod_embeddings'].mean(dim=0).numpy() if len(row['mod_embeddings']) > 0 else np.zeros(row['mod_embeddings'].shape[1])
    pos = row['pos_embeddings'].mean(dim=0).numpy() if len(row['pos_embeddings']) > 0 else np.zeros(row['pos_embeddings'].shape[1])
    
    # Concatena tutti
    return np.concatenate([agent, mod, pos])

# Applica a tutto il dataframe
X_concat = np.vstack(df.apply(mean_concat_embeddings, axis=1))

## Media di tutti gli embeddings insieme

In [6]:
def mean_all_embeddings(row):
    all_embs = []
    for key in ['agent_embeddings','patient_embeddings','mod_embeddings','pos_embeddings']:
        if len(row[key]) > 0:
            all_embs.append(row[key])
    if all_embs:
        return torch.cat(all_embs, dim=0).mean(dim=0).numpy()
    else:
        return np.zeros(row['agent_embeddings'].shape[1])  # fallback
     
X_mean = np.vstack(df.apply(mean_all_embeddings, axis=1))

## Addestramento SVM con cross-validation per type

In [7]:
y = df['Gender'].values

In [8]:
# Standardizziamo sempre prima
scaler_concat = StandardScaler()
X_concat_scaled = scaler_concat.fit_transform(X_concat)

scaler_mean = StandardScaler()
X_mean_scaled = scaler_mean.fit_transform(X_mean)

# Creiamo SVM
svm = SVC(kernel='sigmoid', C=1)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

score_concat = cross_val_score(svm, X_concat_scaled, y, cv=cv)
score_mean = cross_val_score(svm, X_mean_scaled, y, cv=cv)

print("Accuracy SVM (media concatenata agent/mod/pos):", score_concat.mean())
print("Accuracy SVM (media totale embeddings):", score_mean.mean())

Accuracy SVM (media concatenata agent/mod/pos): 0.7801668056713928
Accuracy SVM (media totale embeddings): 0.749024186822352


In [42]:
from sklearn.model_selection import cross_validate

scoring = ['precision', 'recall', 'f1']

results = cross_validate(svm, X_concat_scaled, y, cv=cv, scoring=scoring)

print("Precision:", results['test_precision'].mean())
print("Recall:", results['test_recall'].mean())
print("F1:", results['test_f1'].mean())

Precision: 0.7645962732919255
Recall: 0.7331632653061224
F1: 0.7479425837320574


In [43]:
from sklearn.model_selection import cross_validate

scoring = ['precision', 'recall', 'f1']

results = cross_validate(svm, X_mean_scaled, y, cv=cv, scoring=scoring)

print("Precision:", results['test_precision'].mean())
print("Recall:", results['test_recall'].mean())
print("F1:", results['test_f1'].mean())

Precision: 0.7254109550501007
Recall: 0.7002551020408163
F1: 0.7119191138993454


In [10]:
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm

In [11]:
# Random Forest
# Creiamo Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def rf_cross_val(X, y, model, cv):
    scores = []
    for train_idx, test_idx in tqdm(cv.split(X, y), total=cv.get_n_splits(), desc="Cross-validation RF"):
        model.fit(X[train_idx], y[train_idx])
        score = model.score(X[test_idx], y[test_idx])
        scores.append(score)
    return np.array(scores)

score_concat = rf_cross_val(X_concat, y, rf, cv)
score_mean = rf_cross_val(X_mean, y, rf, cv)

print("Accuracy RF (media concatenata agent//mod/pos):", score_concat.mean())
print("Accuracy RF (media totale embeddings):", score_mean.mean())

Cross-validation RF: 100%|██████████| 5/5 [00:04<00:00,  1.18it/s]

Accuracy RF (media concatenata agent//mod/pos): 0.7746955796497081
Accuracy RF (media totale embeddings): 0.7619015846538784


In [12]:
from sklearn.model_selection import cross_validate

scoring = ['precision', 'recall', 'f1']

results = cross_validate(rf, X_mean_scaled, y, cv=cv, scoring=scoring)

print("Precision:", results['test_precision'].mean())
print("Recall:", results['test_recall'].mean())
print("F1:", results['test_f1'].mean())

Precision: 0.7602586576422172
Recall: 0.6798469387755102
F1: 0.7167968828312279


## Addestramento SVM con cross-validation per Gender

In [13]:
y = df['type'].values

In [48]:
# Standardizziamo sempre prima
scaler_concat = StandardScaler()
X_concat_scaled = scaler_concat.fit_transform(X_concat)

scaler_mean = StandardScaler()
X_mean_scaled = scaler_mean.fit_transform(X_mean)

# Creiamo SVM
svm = SVC(kernel='sigmoid', C=1)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

score_concat = cross_val_score(svm, X_concat_scaled, y, cv=cv)
score_mean = cross_val_score(svm, X_mean_scaled, y, cv=cv)

print("Accuracy SVM (media concatenata agent/mod/pos):", score_concat.mean())
print("Accuracy SVM (media totale embeddings):", score_mean.mean())
from sklearn.model_selection import cross_validate

scoring = ['precision', 'recall', 'f1']

results = cross_validate(svm, X_concat_scaled, y, cv=cv, scoring=scoring)

print("Precision:", results['test_precision'].mean())
print("Recall:", results['test_recall'].mean())
print("F1:", results['test_f1'].mean())
from sklearn.model_selection import cross_validate

scoring = ['precision', 'recall', 'f1']

results = cross_validate(svm, X_mean_scaled, y, cv=cv, scoring=scoring)

print("Precision:", results['test_precision'].mean())
print("Recall:", results['test_recall'].mean())
print("F1:", results['test_f1'].mean())

Accuracy SVM (media concatenata agent/mod/pos): 0.6649207673060884
Accuracy SVM (media totale embeddings): 0.6703753127606339
Precision: 0.6598827477514198
Recall: 0.6788552188552189
F1: 0.6675954262092876
Precision: 0.6722094672019654
Recall: 0.664040404040404
F1: 0.6658954454091586


In [15]:
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
# Random Forest
# Creiamo Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def rf_cross_val(X, y, model, cv):
    scores = []
    for train_idx, test_idx in tqdm(cv.split(X, y), total=cv.get_n_splits(), desc="Cross-validation RF"):
        model.fit(X[train_idx], y[train_idx])
        score = model.score(X[test_idx], y[test_idx])
        scores.append(score)
    return np.array(scores)

score_concat = rf_cross_val(X_concat, y, rf, cv)
score_mean = rf_cross_val(X_mean, y, rf, cv)

print("Accuracy RF (media concatenata agent//mod/pos):", score_concat.mean())
print("Accuracy RF (media totale embeddings):", score_mean.mean())
from sklearn.model_selection import cross_validate

scoring = ['precision', 'recall', 'f1']

results = cross_validate(rf, X_mean_scaled, y, cv=cv, scoring=scoring)

print("Precision:", results['test_precision'].mean())
print("Recall:", results['test_recall'].mean())
print("F1:", results['test_f1'].mean())

Cross-validation RF: 100%|██████████| 5/5 [00:04<00:00,  1.17it/s]


Accuracy RF (media concatenata agent//mod/pos): 0.6594495412844037
Accuracy RF (media totale embeddings): 0.661234361968307
Precision: 0.6800053941031886
Recall: 0.6198653198653198
F1: 0.6457951510149014
